# Data loading

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("CollegeScores4yr.csv")
df.dropna(subset=['CompRate'], inplace=True)
df.drop(columns=['Accred', 'State', 'MainDegree'], inplace=True)
cols_with_na = ['AdmitRate', 'MidACT', 'AvgSAT']
na_flags = df[cols_with_na].isna().astype(int).add_suffix('_na')
df = pd.concat([df, na_flags], axis=1)
df.head()

,Name,ID,Main,HighDegree,Control,Region,Locale,Latitude,Longitude,AdmitRate,...,InstructFTE,FacSalary,FullTimeFac,Pell,CompRate,Debt,PctWomen,AdmitRate_na,MidACT_na,AvgSAT_na
0,Alabama A & M University,100654,1,4,Public,Southeast,City,34.783368,-86.568502,68.4,...,7073.0,8651.0,70.6,65.4,26.8,31000.0,59.5,0,0,0
1,University of Alabama at Birmingham,100663,1,4,Public,Southeast,City,33.505697,-86.799345,86.7,...,15256.0,11837.0,75.3,33.1,64.4,22300.0,62.5,0,0,0
2,Amridge University,100690,1,4,Private,Southeast,City,32.362609,-86.174010,NaN,...,5927.0,4134.0,100.0,77.7,50.0,32189.0,63.6,1,1,1
3,University of Alabama in Huntsville,100706,1,4,Public,Southeast,City,34.724557,-86.640449,78.1,...,8733.0,10267.0,65.0,21.7,62.9,20705.0,40.2,0,0,0
4,Alabama State University,100724,1,4,Public,Southeast,City,32.364317,-86.295677,96.6,...,9349.0,8071.0,69.6,69.8,27.7,31000.0,64.0,0,0,0


# Part I: Tree-based methods

In [3]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=598)
meta_train = train_df[['ID', 'Name']]
meta_test = test_df[['ID', 'Name']]

X_train = train_df.drop(columns=['CompRate', 'ID', 'Name']).copy()
y_train = train_df['CompRate'].copy()

X_test = test_df.drop(columns=['CompRate', 'ID', 'Name']).copy()
y_test = test_df['CompRate'].copy()

# Fill numerical n/a with median
medians = X_train.median(numeric_only=True)
X_train.fillna(medians, inplace=True)
X_test.fillna(medians, inplace=True)

# Fill categorical n/a with mode
modes = X_train.select_dtypes(include='object').mode().iloc[0]
X_train.fillna(modes, inplace=True)
X_test.fillna(modes, inplace=True)

# Map any unseen categories in test to training mode
for col in modes.index:
    train_levels = set(X_train[col].unique())
    mode_val = modes[col]
    X_test[col] = X_test[col].apply(lambda x: x if x in train_levels else mode_val)

/var/folders/n5/wz31rvqd33q6chtntps3fjn40000gn/T/ipykernel_51684/378825171.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  modes = X_train.select_dtypes(include='object').mode().iloc[0]
